In [1]:
from google.colab import drive
drive.mount("/content/drive")

proj_path = '/content/drive/MyDrive/CSE 3521/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# right click to upload transformer.py and dataset.py to colab runtime

import sys
sys.path.append('/content')

from fine_tune_dataset import ReviewDataset
from utils import report_metrices

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

In [4]:
model_name = 'bert-base-uncased'
num_labels = 3

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
for p in model.parameters(): p.requires_grad = False
for layer in model.bert.encoder.layer[-4:]:
    for p in layer.parameters(): p.requires_grad = True
for p in model.bert.pooler.parameters():
    p.requires_grad = True
for p in model.classifier.parameters():
    p.requires_grad = True

In [6]:
trainable = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
enc  = [p for n, p in trainable if n.startswith("bert.")]
head = [p for n, p in trainable if n.startswith("classifier.")]

optimizer = torch.optim.AdamW([
    {"params": [p for p in enc if p.ndim > 1],  "lr": 2e-5, "weight_decay": 0.01},
    {"params": [p for p in enc if p.ndim <= 1], "lr": 2e-5, "weight_decay": 0.0},
    {"params": head,                            "lr": 1e-3, "weight_decay": 0.0},
])

In [7]:
training_args = TrainingArguments(
    learning_rate=2e-5,                
    num_train_epochs=3,
    per_device_train_batch_size=32,    
    per_device_eval_batch_size=64,

    weight_decay=0.01,                 
    warmup_steps=500,

    dataloader_num_workers=2,

    eval_strategy="steps",
    eval_steps=250,
    logging_steps=50,
    logging_first_step=True,

    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

)

In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

In [9]:
train_dataset = ReviewDataset(proj_path + "data/train.csv", tokenizer)
val_dataset = ReviewDataset(proj_path + "data/val.csv", tokenizer)

In [10]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    compute_metrics=compute_metrics,
    optimizers = (optimizer, None)
)


In [11]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy,Macro F1
250,0.621587,0.566262,0.761226,0.667173
500,0.522097,0.499134,0.792098,0.691434
750,0.464882,0.488404,0.803972,0.721476
1000,0.491548,0.456487,0.807642,0.735529
1250,0.404798,0.471408,0.803756,0.741258
1500,0.444918,0.437832,0.821028,0.755575
1750,0.441307,0.439982,0.815199,0.751576
2000,0.393361,0.443379,0.817789,0.756835
2250,0.396654,0.448945,0.822323,0.758532
2500,0.384339,0.447052,0.817573,0.756177


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (6 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=3042, training_loss=0.4588678005022253, metrics={'train_runtime': 3588.9163, 'train_samples_per_second': 27.098, 'train_steps_per_second': 0.848, 'total_flos': 1.2794416163859456e+16, 'train_loss': 0.4588678005022253, 'epoch': 3.0})

In [12]:
trainer.save_model(proj_path + "modern/fine_tuned")
tokenizer.save_pretrained(proj_path+"modern/fine_tuned")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/CSE 3521/modern/fine_tuned/tokenizer_config.json',
 '/content/drive/MyDrive/CSE 3521/modern/fine_tuned/tokenizer.json')

In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_dir = proj_path + "modern/fine_tuned"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

In [14]:
import torch, torch.nn.functional as F

@torch.inference_mode()
def predict_batch(texts, model, tokenizer, device, batch_size=64, max_length=256):
    model.eval()
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = tokenizer(
            texts[i:i + batch_size],
            truncation=True, max_length=max_length,
            padding=True, return_tensors="pt",
        ).to(device)
        logits = model(**batch).logits
        all_probs.append(logits.softmax(dim=-1).cpu())
    return torch.cat(all_probs, dim=0)   

In [15]:
import pandas as pd
df = pd.read_csv(proj_path + "data/test.csv")
texts = df["input_text"].astype(str).tolist()
probs = predict_batch(texts, model, tokenizer, device)
y_pred = probs.argmax(dim=-1).tolist()

report_metrices(df['Sentiment'], y_pred)


F1         per-class [0.8177935943060498, 0.59683578832515, 0.9010788381742738]  macro 0.772  micro 0.828  weighted 0.823
Precision  per-class [0.7962577962577962, 0.6590361445783133, 0.8813311688311688]  macro 0.779  micro 0.828  weighted 0.821
Recall     per-class [0.8405267008046818, 0.5453639082751744, 0.9217317487266553]  macro 0.769  micro 0.828  weighted 0.828
Accuracy   0.828
Confusion Matrix  labels [0, 1, 2]
[[1149  173   45]
 [ 226 1094  686]
 [  68  393 5429]]


{'accuracy': 0.8282413904782467,
 'f1': {'per_class': [0.8177935943060498,
   0.59683578832515,
   0.9010788381742738],
  'macro': 0.7719027402684911,
  'micro': 0.8282413904782467,
  'weighted': 0.8229008735445421},
 'precision': {'per_class': [0.7962577962577962,
   0.6590361445783133,
   0.8813311688311688],
  'macro': 0.7788750365557595,
  'micro': 0.8282413904782467,
  'weighted': 0.8206360248217734},
 'recall': {'per_class': [0.8405267008046818,
   0.5453639082751744,
   0.9217317487266553],
  'macro': 0.7692074526021705,
  'micro': 0.8282413904782467,
  'weighted': 0.8282413904782467},
 'labels': [0, 1, 2],
 'confusion_matrix': [[1149, 173, 45], [226, 1094, 686], [68, 393, 5429]]}